# ACC7013 — LAB 

### Predictive Analytics II: Machine Learning Classification

In [ ]:
!pip install scikit-learn==1.5.2 
!pip install statsmodels==0.14.4

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

import statsmodels.formula.api as smf

np.random.seed(42) # for replication

### PART 1 — load the ols_logit.csv dataset from PracticeData folder 

In [ ]:
#df.head()
#df.info()
#df.describe()

### PART 2 — Logistic Regression Baseline 

In [ ]:
# NOTE: CreditScore is excluded — it would perfectly predict Default

logit_model = smf.logit("Default ~ Income + DebtRatio + Age", data=df).fit()
print(logit_model.summary())

In [ ]:
# Odds ratios
odds_ratios = pd.DataFrame({
    "Odds Ratio": logit_model.params.apply(np.exp),
    "p-value":    logit_model.pvalues
})
print("\nOdds Ratios:")
print(odds_ratios)

In [ ]:
# Marginal effects
mfx = logit_model.get_margeff()
print("\nMarginal Effects:")
print(mfx.summary())

### PART 3 — Train/Test Split

In [ ]:
X = df[["Income", "DebtRatio", "Age"]]
y = df["Default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

### PART 4 — Feature Scaling (required for k-NN)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

### PART 5 — k-NN: Comparing Multiple k Values

In [ ]:
for k in [3, 5, 7, 9]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred_knn = knn.predict(X_test_scaled)
    print(f"\n=== k-NN  (k={k}) ===")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_knn))
    print(classification_report(y_test, y_pred_knn))

### PART 6 — k-NN: Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {"n_neighbors": [3, 5, 7, 9, 11]}
grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
grid.fit(X_train_scaled, y_train)

print("\n=== GridSearchCV: Best k-NN Parameters ===")
print("Best k:        ", grid.best_params_)
print("Best CV Score: ", round(grid.best_score_, 4))

# Use the best k-NN going forward
best_knn = grid.best_estimator_

### PART 7 — Decision Tree

In [ ]:
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

print("\n=== Decision Tree ===")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

plt.figure(figsize=(12, 6))
plot_tree(tree, feature_names=X.columns,
          class_names=["No Default", "Default"], filled=True)
plt.title("Decision Tree (max_depth=4)")
plt.show()

### PART 8 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest ===")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

print("\nFeature Importances:")
print(pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False))

### PART 9 — Cross-Validation (Random Forest)

In [ ]:
cv_scores_rf = cross_val_score(rf, X, y, cv=5)
print("\nRandom Forest CV Scores:", cv_scores_rf)
print("Mean CV Score:          ", round(cv_scores_rf.mean(), 4))


### PART 10 — ROC Curve Comparison (All Models)

In [ ]:
plt.figure(figsize=(7, 5))

# Logistic Regression (statsmodels — full dataset probabilities)
y_prob_logit = logit_model.predict(df)
fpr_logit, tpr_logit, _ = roc_curve(y, y_prob_logit)
auc_logit = auc(fpr_logit, tpr_logit)
plt.plot(fpr_logit, tpr_logit, label=f"Logit (AUC={auc_logit:.3f})")

# k-NN (best k, test set)
y_prob_knn = best_knn.predict_proba(X_test_scaled)[:, 1]
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_prob_knn)
auc_knn = auc(fpr_knn, tpr_knn)
plt.plot(fpr_knn, tpr_knn, label=f"k-NN k={grid.best_params_['n_neighbors']} (AUC={auc_knn:.3f})")

# Decision Tree (test set)
y_prob_tree = tree.predict_proba(X_test)[:, 1]
fpr_tree, tpr_tree, _ = roc_curve(y_test, y_prob_tree)
auc_tree = auc(fpr_tree, tpr_tree)
plt.plot(fpr_tree, tpr_tree, label=f"Decision Tree (AUC={auc_tree:.3f})")

# Random Forest (test set)
y_prob_rf = rf.predict_proba(X_test)[:, 1]
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
auc_rf = auc(fpr_rf, tpr_rf)
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={auc_rf:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: Logit vs ML Models")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()




### PART 11 — Student Tasks

In [ ]:
print("""
===========================
STUDENT TASKS
===========================

1.  Change k in k-NN (try 3, 7, 11) and compare precision/recall.
2.  Change max_depth in the Decision Tree — when does it overfit?
3.  Change n_estimators in Random Forest and compare AUC.
4.  Interpret the feature importances — which variable matters most?
5.  Does scaling the data affect Decision Tree performance? Why / why not?
6.  Recreate the ROC curves and add the k-NN line yourself.
7.  Compare all ML models against logistic regression in a summary table.
8.  Interpret the odds ratios from the statsmodels logit output.
9.  Interpret the marginal effects — which variable has the strongest effect?
10. Write a short paragraph: which model performs best and why?
""")
